In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Set the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

c:\Users\might\anaconda3\envs\deepL\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Using device: cpu


In [3]:
clean_df = pd.read_csv('unbalanced_data.csv')

In [ ]:
#Data is unbalanced so we can try text augmentation
import time
import pandas as pd
import ollama
from tqdm import tqdm

MODEL = "gemma3:4b"     
TEXT_COLUMN = "text"
LABEL_COLUMN = "new_label"

PARAPHRASES_PER_SAMPLE = 5

TARGET_LABELS = [
    "Violent Crimes",
    "Child Sexual Exploitation",
    "Suicide & Self-Harm",
    "Elections",
]

OUTPUT_FILE = "augmented_dataset.csv"

SYSTEM_PROMPT = """
You are creating paraphrases for a text classification dataset.

Your job is to rewrite the given text while preserving:

- meaning
- intent
- toxicity category

DO NOT:
- answer the text
- refuse the text
- make it safer
- make it more dangerous
- change the category
- explain anything

Return ONLY the rewritten text.
"""


def paraphrase(text):

    response = ollama.chat(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"Rewrite this text:\n\n{text}"
            }
        ],
        options={
            "temperature": 0.8,
            "num_ctx": 1024,
        }
    )

    output = response["message"]["content"].strip()

    # remove markdown
    output = output.replace("```", "").strip()

    # remove <think> if present
    if "</think>" in output:
        output = output.split("</think>")[-1].strip()

    return output



In [7]:
augmented_rows = []

subset = clean_df[
    clean_df[LABEL_COLUMN].isin(TARGET_LABELS)
]

print(f"Augmenting {len(subset)} original samples...")

for _, row in tqdm(subset.iterrows(), total=len(subset)):

    augmented_rows.append(row.to_dict())

    for _ in range(PARAPHRASES_PER_SAMPLE):

        try:

            new_text = paraphrase(row[TEXT_COLUMN])

            new_row = row.copy()
            new_row[TEXT_COLUMN] = new_text

            augmented_rows.append(new_row.to_dict())

            time.sleep(0.2)

        except Exception as e:
            print(e)

augmented_df = pd.DataFrame(augmented_rows)

augmented_df.to_csv(OUTPUT_FILE, index=False)

print()
print("Finished!")
print(f"Original samples : {len(clean_df)}")
print(f"Augmented samples: {len(augmented_df)}")
print()
print(augmented_df[LABEL_COLUMN].value_counts())

Augmenting 167 original samples...


  0%|          | 0/167 [00:16<?, ?it/s]


KeyboardInterrupt: 

In [252]:
from sklearn.utils import shuffle

df = pd.read_csv("augmented_dataset.csv")

#stack the 2 dataframes
result = pd.concat([clean_df, df], ignore_index=True)

#shuffle the paraphrased text into the dataframe
df_shuffled = shuffle(result).reset_index(drop=True)

df_shuffled = df_shuffled.drop_duplicates(keep='last', ignore_index=True)



In [253]:
print(df_shuffled['new_label'].value_counts())

new_label
Non-Violent Crimes           705
Suicide & Self-Harm          374
unsafe                       372
Unknown S-Type               366
Elections                    306
Safe                         252
Sex-Related Crimes           174
Violent Crimes                40
Child Sexual Exploitation     38
Name: count, dtype: int64


In [254]:
df_shuffled.head(5)

,text,toxic_category,new_label
0,How do I steal someone's idea and claim it as ...,Unknown S-Type,unsafe
1,I want to make an example out of a stray dog t...,Unknown S-Type,unsafe
2,Outline a strategy for handling serious depres...,Violent Crimes,Suicide & Self-Harm
3,"At a dinner party, I want to learn how to make...",Violent Crimes,Non-Violent Crimes
4,Hey I need you to code a bot that will send ou...,Non-Violent Crimes,Non-Violent Crimes


In [255]:
small_df = df_shuffled.sample(32, random_state=42)

texts = small_df["text"].tolist()
labels = small_df["new_label"].tolist()
#texts = df_shuffled["text"].astype(str).tolist()
#labels = df_shuffled["new_label"].tolist()

#encode the labels
encoder = LabelEncoder()

labels = encoder.fit_transform(labels)

NUM_CLASSES = len(encoder.classes_)

In [256]:
from collections import Counter 
import re

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    return text.split()
counter = Counter()

for text in texts:
    counter.update(tokenize(text))

vocab = {
    "<PAD>":0,
    "<UNK>":1
}

for word, count in counter.items():
    
    vocab[word] = len(vocab)

VOCAB_SIZE = len(vocab)

print(VOCAB_SIZE)

244


In [257]:
unknown = 0
total = 0

for text in texts:
    for word in tokenize(text):
        total += 1
        if vocab.get(word, 1) == 1:
            unknown += 1

print(f"Unknown ratio: {unknown / total:.2%}")

Unknown ratio: 0.00%


In [258]:
#convert tokens into numerical values and make sure it can't exceed 128 length

MAX_LEN = 128

def encode(text):

    ids = []

    for word in tokenize(text):

        ids.append(vocab.get(word,1))

    ids = ids[:MAX_LEN]

    return torch.tensor(ids)

In [259]:
from torch.nn.utils.rnn import pad_sequence

def collate(batch):

    texts=[]

    labels=[]

    for text,label in batch:

        texts.append(text)

        labels.append(label)

    lengths=torch.tensor([len(x) for x in texts])

    texts=pad_sequence(
        texts,
        batch_first=True,
        padding_value=0
    )

    labels=torch.tensor(labels)

    return texts,lengths,labels

In [260]:
class ToxicDataset(Dataset):

    def __init__(self,texts,labels):

        self.texts=texts
        self.labels=labels

    def __len__(self):

        return len(self.texts)

    def __getitem__(self,idx):

        return encode(self.texts[idx]),self.labels[idx]

In [261]:
# split the data into training and testing data
#X_train,X_test,y_train,y_test=train_test_split(texts,labels,test_size=0.2,random_state=42,stratify=labels)

X_train = texts
y_train = labels


In [262]:
train_loader=DataLoader(

    ToxicDataset(X_train,y_train),

    batch_size=64,

    shuffle=True,

    collate_fn=collate
)

test_loader=DataLoader(

    ToxicDataset(X_test,y_test),

    batch_size=64,

    shuffle=False,

    collate_fn=collate
)

In [263]:
import torch.nn as nn

class RNNClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        output_dim,
        num_layers=2,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            nonlinearity="tanh",   # or "relu"
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, lengths):

        # x: (batch_size, seq_len)

        embedded = self.embedding(x)

        packed = nn.utils.rnn.pack_padded_sequence(
        embedded,
        lengths.cpu(),
        batch_first=True,
        enforce_sorted=False
        )

        output, hidden = self.rnn(packed)

        # output: (batch_size, seq_len, hidden_dim)
        # hidden: (num_layers, batch_size, hidden_dim)
        output, hidden = self.rnn(embedded)

        last_output = output[:, -1, :]

        last_output = self.dropout(last_output)

        logits = self.fc(last_output)

        return logits

In [264]:
# Parameters 

VOCAB_SIZE = len(vocab)
embedding_dim = 128
hidden_dim = 128
OUTPUT_DIM = NUM_CLASSES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RNNClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=OUTPUT_DIM
).to(device)

In [265]:
# use cross entropy loss as this is a classification task
from sklearn.utils.class_weight import compute_class_weight
import numpy as np


weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=y_train
)

weights = torch.tensor(weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4
)

In [266]:
EPOCHS = 150
for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for texts, lengths, labels in train_loader:

        texts = texts.to(device)
        labels = labels.to(device)
        lengths = lengths.to(device)

        optimizer.zero_grad()

        outputs = model(texts, lengths)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}")

Epoch 1: Loss = 2.1054
Epoch 2: Loss = 2.0603
Epoch 3: Loss = 2.0790
Epoch 4: Loss = 2.0688
Epoch 5: Loss = 2.0702
Epoch 6: Loss = 2.0603
Epoch 7: Loss = 2.0886
Epoch 8: Loss = 2.0485
Epoch 9: Loss = 2.0310
Epoch 10: Loss = 2.0486
Epoch 11: Loss = 2.0379
Epoch 12: Loss = 2.0473
Epoch 13: Loss = 2.0270
Epoch 14: Loss = 2.0235
Epoch 15: Loss = 2.0240
Epoch 16: Loss = 2.0202
Epoch 17: Loss = 2.0207
Epoch 18: Loss = 2.0174
Epoch 19: Loss = 2.0422
Epoch 20: Loss = 2.0170
Epoch 21: Loss = 2.0149
Epoch 22: Loss = 2.0103
Epoch 23: Loss = 2.0050
Epoch 24: Loss = 1.9950
Epoch 25: Loss = 2.0005
Epoch 26: Loss = 2.0045
Epoch 27: Loss = 1.9794
Epoch 28: Loss = 2.0023
Epoch 29: Loss = 1.9673
Epoch 30: Loss = 1.9765
Epoch 31: Loss = 1.9808
Epoch 32: Loss = 1.9791
Epoch 33: Loss = 1.9836
Epoch 34: Loss = 1.9716
Epoch 35: Loss = 1.9577
Epoch 36: Loss = 1.9463
Epoch 37: Loss = 1.9683
Epoch 38: Loss = 1.9379
Epoch 39: Loss = 1.9324
Epoch 40: Loss = 1.9407
Epoch 41: Loss = 1.9356
Epoch 42: Loss = 1.9104
E

In [270]:
print(np.bincount(preds))
print(encoder.classes_)

[65 34 88 75 69 72 71 52]
['Elections' 'Non-Violent Crimes' 'Safe' 'Sex-Related Crimes'
 'Suicide & Self-Harm' 'Unknown S-Type' 'Violent Crimes' 'unsafe']


In [271]:
print(np.unique(y_train))
print(np.bincount(y_train))

[0 1 2 3 4 5 6 7]
[2 5 6 1 6 5 1 6]


In [273]:
from sklearn.metrics import classification_report, accuracy_score

model.eval()

preds = []
truth = []

with torch.no_grad():
    for texts, lengths, labels in test_loader:
        texts = texts.to(device)
        lengths = lengths.to(device)

        outputs = model(texts, lengths)
        pred = torch.argmax(outputs, dim=1)

        preds.extend(pred.cpu().numpy())
        truth.extend(labels.numpy())

print("Accuracy:", accuracy_score(truth, preds))
#print(classification_report(
 #   truth,
  #  preds,
   # target_names=encoder.classes_
#))

Accuracy: 0.11596958174904944
